In [58]:
from pathlib import Path
from typing import Dict, List, Tuple

def load_intervals(path: Path) -> Dict[str, List[Tuple[float, float]]]:
    """
    Read anomaly intervals from `train-anomaly-results.txt`.

    Expected line format (whitespace-separated):
        <video_id:int> <start:int> <end:int>

    Returns:
        dict mapping "<video_id>" (without extension) -> list of (start, end) intervals as floats.
    """
    intervals: Dict[str, List[Tuple[float, float]]] = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) != 3:
                continue  # skip malformed rows
            vid, s, e = parts
            key = vid.strip()  # e.g., "6"
            try:
                s_f, e_f = float(s), float(e)
            except ValueError:
                continue
            intervals.setdefault(key, []).append((s_f, e_f))
    return intervals


def time_in_any_interval(t: float, ivs: List[Tuple[float, float]]) -> bool:
    """Return True iff t is within any [start, end] interval (inclusive)."""
    for s, e in ivs:
        if s <= t <= e:
            return True
    return False


def evaluate_detections(
    intervals_by_video: Dict[str, List[Tuple[float, float]]],
    results_path: Path,
) -> Tuple[int, int, int, int, float]:
    """
    Evaluate detections using ad_results.txt.

    Expected line format (whitespace-separated):
        <video_filename:str> <time_in_seconds:float> <label: 'YES'|'NO'>

    Returns:
        (TP, FP, TN, FN, accuracy)
    """
    TP = FP = TN = FN = 0

    with results_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 3:
                continue  # skip malformed
            filename, t_str, label = parts[0], parts[1], parts[2]
            # Normalise video key by stripping extension and any directories
            base = Path(filename).name
            key = base.rsplit(".", 1)[0]  # "6.mp4" -> "6"

            try:
                t = float(t_str)
            except ValueError:
                continue

            label = label.upper()
            ivs = intervals_by_video.get(key, [])  # empty if no anomalies known for this video
            is_anom = time_in_any_interval(t, ivs)

            if label == "YES":
                if is_anom:
                    TP += 1
                else:
                    FP += 1
                    #print(f"FP detected @ {filename}")
            elif label == "NO":
                if is_anom:
                    FN += 1
                    #print(f"FN detected @ {filename}")
                else:
                    TN += 1
            else:
                # Unknown label; skip silently or log if desired
                continue
                

    total = TP + FP + TN + FN
    accuracy = (TP + TN) / total if total else 0.0
    prec = TP / (TP + FP)
    rec = TP / (TP + FN)
    f1 = 2 * (prec * rec) / (prec + rec)
    return TP, FP, TN, FN, accuracy, prec, rec, f1


if __name__ == "__main__":
    frame_slot = 360
    yolo_conf = 0.6
    
    # Adjust paths if your working directory differs
    intervals_path = Path("../../Demo_AD/aic21-track4-train-data/train-anomaly-results.txt")
    results_path = Path(f"ad_results_{frame_slot}_conf{str(yolo_conf).split('.')[-1]}.txt")

    intervals_by_video = load_intervals(intervals_path)
    TP, FP, TN, FN, acc, prec, rec, f1 = evaluate_detections(intervals_by_video, results_path)

    print(f"#frameSlot = {frame_slot}, confidence = 0.{yolo_conf}")
    print(f"TP: {TP}")
    print(f"FP: {FP}")
    print(f"TN: {TN}")
    print(f"FN: {FN}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1: {f1:.4f}")

    # Tabella frame_slot
    #360 & 0.9040 & 0.5577 & 0.5963 & 0.5763 \\
    #print(f"{frame_slot} & {acc:.4f} & {prec:.4f} & {rec:.4f} & {f1:.4f}")

    # Tabella confidence
    # 0.30 & 0.8310 & 0.3703 & \textbf{0.7692} & 0.5000 \\
    print(f"{yolo_conf} & {acc:.4f} & {prec:.4f} & {rec:.4f} & {f1:.4f}")
    


#frameSlot = 360, confidence = 0.0.6
TP: 419
FP: 231
TN: 6325
FN: 391
Accuracy: 0.9156
Precision: 0.6446
Recall: 0.5173
F1: 0.5740
0.6 & 0.9156 & 0.6446 & 0.5173 & 0.5740


In [ ]:
#frameSlot = 120, confidence = 0.55
TP: 1446
FP: 1063
TN: 18486
FN: 971
Accuracy: 0.907402

#frameSlot = 240, confidence = 0.55
TP: 715
FP: 555
TN: 9238
FN: 495
Accuracy: 0.904571

#frameSlot = 360, confidence = 0.55
TP: 483
FP: 383
TN: 6173
FN: 327
Accuracy: 0.903611

#frameSlot = 480, confidence = 0.55
TP: 350
FP: 271
TN: 4632
FN: 254
Accuracy: 0.904667

#frameSlot = 600, confidence = 0.55
TP: 283
FP: 222
TN: 3719
FN: 202
Accuracy: 0.904202

In [15]:
#frameSlot = 120
TP: 1835
FP: 2670
TN: 16879
FN: 582
Accuracy: 0.851953

#frameSlot = 240
TP: 880
FP: 1244
TN: 8549
FN: 330
Accuracy: 0.856948

#frameSlot = 360
TP: 589
FP: 845
TN: 5711
FN: 221
Accuracy: 0.855281

#frameSlot = 360, confidence = 0.3
TP: 623
FP: 1059
TN: 5497
FN: 187
Accuracy: 0.830844

#frameSlot = 360, confidence = 0.45
TP: 542
FP: 595
TN: 5961
FN: 268
Accuracy: 0.882840

#frameSlot = 360, confidence = 0.5
TP: 517
FP: 478
TN: 6078
FN: 293
Accuracy: 0.895330

#frameSlot = 360, confidence = 0.55
TP: 483
FP: 383
TN: 6173
FN: 327
Accuracy: 0.903611

#frameSlot = 360, confidence = 0.6
TP: 419
FP: 231
TN: 6325
FN: 391
Accuracy: 0.915558

#frameSlot = 480
TP: 428
FP: 636
TN: 4267
FN: 176
Accuracy: 0.852551


In [24]:
# Parametri per l'elaborazione
frameSlot = 360   # numero di frame da usare per calcolare il background
timeSlot = 3600   # distanza tra due analisi successive
total_duration = 12  # secondi
yolo_conf = 0.5

# Percorso della cartella contenente i video
video_dir = "../../Demo_AD/aic21-track4-train-data/"



# --- CONFIG: path output ---
OUTPUT_CSV = "ad_iterations_timings.csv"
OUTPUT_RESULTS = f"ad_results_{str(frameSlot)}_conf{str(yolo_conf).split('.')[-1]}.txt"
print(OUTPUT_RESULTS)

ad_results_360_conf5.txt
